# TrashNet exploratory data analysis

This notebook inspects class balance, image dimensions, and representative samples before model training. Run it from the repository root after downloading the dataset.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
from PIL import Image

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT_DIR / 'data' / 'trashnet' / 'raw'
CLASSES = ('cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

## Class distribution

TrashNet is imbalanced: the general-trash category is much smaller than the recyclable categories. Stratified train, validation, and test splits preserve this distribution.

In [ ]:
class_files = {
    label: [path for path in (DATA_DIR / label).iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS]
    for label in CLASSES
}
class_counts = {label: len(paths) for label, paths in class_files.items()}
class_counts

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ['#B7794B', '#45A99A', '#718096', '#D7A93E', '#4A86E8', '#7B6F83']
bars = ax.bar(class_counts.keys(), class_counts.values(), color=colors)
ax.bar_label(bars, padding=3)
ax.set(title='TrashNet class distribution', xlabel='Material', ylabel='Images')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

## Image dimensions

The training pipeline resizes every image to 224 × 224 pixels. This check confirms the range of original aspect ratios and resolutions.

In [ ]:
dimensions = []
for label, paths in class_files.items():
    for path in paths:
        with Image.open(path) as image:
            dimensions.append({'class': label, 'width': image.width, 'height': image.height})

widths = [item['width'] for item in dimensions]
heights = [item['height'] for item in dimensions]
{
    'images': len(dimensions),
    'width_range': (min(widths), max(widths)),
    'height_range': (min(heights), max(heights)),
}

## Representative samples

A small visual sample helps reveal background, lighting, and viewpoint variation that the model needs to handle.

In [ ]:
random.seed(42)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for axis, label in zip(axes.flat, CLASSES):
    sample_path = random.choice(class_files[label])
    with Image.open(sample_path) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(label.title())
    axis.axis('off')

fig.suptitle('One sample per TrashNet class', fontsize=16)
plt.tight_layout()
plt.show()

## Modeling implications

- Use stratified splits so the small trash class appears in every split.
- Apply color and horizontal-flip augmentation to reduce sensitivity to capture conditions.
- Report per-class precision and recall alongside overall accuracy.
- Keep a human-review path for low-confidence predictions and visually similar glass/plastic items.